# M6: 転移学習で自前画像を多クラス分類する

## このノートブックでできること
- 事前学習済み CNN（ResNet など）を **特徴抽出器（Feature Extractor）** として使います
- scikit-learn の LogisticRegression で画像の多クラス分類を行います
- **自前の画像データセット**（Google Drive）を使って自由なテーマを分類できます
- `img2feat` ライブラリによる簡易な特徴抽出方法も紹介します

## 所要時間の目安
約 45〜60 分（データ準備込み）

## 対応するサイトのモジュール
**M6: 転移学習 — 少ないデータで賢く**（授業課題3・最終レポートに直結）

## GPU について
特徴抽出フェーズは GPU があると速いですが、CPU でも動きます。特徴抽出後の分類（LogisticRegression）は CPU で十分です。


## 1. ライブラリのインポートと環境確認


In [ ]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torchvision import models
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import seaborn as sns
from tqdm.auto import tqdm

# 乱数シードの固定
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用デバイス: {device}")
print(f"PyTorch: {torch.__version__}  torchvision: {torchvision.__version__}")


## 2. Google Drive のマウントとデータセットの準備

### データセットのディレクトリ構成

自前の画像データセットは以下のフォルダ構成で準備してください:

```
MyDataset/
├── train/
│   ├── クラスA/
│   │   ├── img001.jpg
│   │   ├── img002.jpg
│   │   └── ...
│   ├── クラスB/
│   │   └── ...
│   └── クラスC/
│       └── ...
└── test/
    ├── クラスA/
    │   └── ...
    ├── クラスB/
    │   └── ...
    └── クラスC/
        └── ...
```

- 画像サイズ: **256×256 ピクセル推奨**（このノートブックで自動リサイズします）
- 対応フォーマット: JPEG / PNG
- 1クラスあたり最低 **10 枚以上**推奨（転移学習なら少ない枚数でも機能します）

### Google Drive へのアップロード手順
1. Google Drive 上に `MyDataset/train/クラス名/` フォルダを作成
2. 各クラスの画像をアップロード
3. 以下のセルを実行して Drive をマウント


In [ ]:
# Google Drive をマウント（Colab 上で実行してください）
# from google.colab import drive
# drive.mount('/content/drive')

# ============================================================
# データセットのパスを設定してください
# ============================================================
# Google Drive を使う場合:
# DATASET_ROOT = '/content/drive/MyDrive/MyDataset'

# このノートブックのデモ用: CIFAR-10 のサブセットを使用
USE_DEMO_DATA = True  # 自前データを使う場合は False に変更

if USE_DEMO_DATA:
    DATASET_ROOT = './demo_dataset'
    print("デモデータモードを使用します（CIFAR-10 から 5 クラスを抽出）")
    print("自前データを使う場合は USE_DEMO_DATA = False にしてください。")
else:
    DATASET_ROOT = '/content/drive/MyDrive/MyDataset'  # 変更してください
    print(f"データセットパス: {DATASET_ROOT}")


## 3. デモデータの作成（自前データを使う場合はスキップ可）

CIFAR-10 から 5 クラスのサブセットを作成して動作確認します。自前データセットを持っている場合は `USE_DEMO_DATA = False` にしてください。


In [ ]:
if USE_DEMO_DATA:
    import torchvision
    from torchvision.datasets import CIFAR10

    # CIFAR-10 をダウンロード
    raw_train = CIFAR10(root='./data', train=True, download=True)
    raw_test  = CIFAR10(root='./data', train=False, download=True)

    # 使用するクラス（5 クラス）
    SELECTED_CLASSES = ['airplane', 'automobile', 'bird', 'cat', 'deer']
    CLASS_INDICES = {c: i for i, c in enumerate(raw_train.classes) if c in SELECTED_CLASSES}

    print(f"使用クラス: {SELECTED_CLASSES}")
    print(f"クラスインデックス: {CLASS_INDICES}")

    def save_subset(dataset, root, classes, n_per_class=50):
        """指定クラスの画像を n_per_class 枚ずつ保存する。"""
        counters = {c: 0 for c in classes}
        for img, label in dataset:
            cls_name = dataset.classes[label]
            if cls_name not in classes:
                continue
            if counters[cls_name] >= n_per_class:
                continue
            save_dir = os.path.join(root, cls_name)
            os.makedirs(save_dir, exist_ok=True)
            # PIL Image を 64×64 に拡大して保存
            img_pil = Image.fromarray(np.array(img)).resize((64, 64), Image.BILINEAR)
            img_pil.save(os.path.join(save_dir, f"{counters[cls_name]:04d}.png"))
            counters[cls_name] += 1
            if all(v >= n_per_class for v in counters.values()):
                break

    save_subset(raw_train, os.path.join(DATASET_ROOT, 'train'), SELECTED_CLASSES, n_per_class=100)
    save_subset(raw_test,  os.path.join(DATASET_ROOT, 'test'),  SELECTED_CLASSES, n_per_class=30)

    # 保存確認
    for split in ['train', 'test']:
        for cls in SELECTED_CLASSES:
            n = len(glob.glob(os.path.join(DATASET_ROOT, split, cls, '*.png')))
            print(f"  {split}/{cls}: {n} 枚")


## 4. 事前学習済みモデルの読み込み

ResNet-50 の事前学習済み重みを使います。**最後の全結合層（分類器）を除いた部分が特徴抽出器**として機能します。

転移学習（Transfer Learning）のイメージ:
- ImageNet（1,000 クラス・100 万枚以上）で学習した「汎用的な視覚特徴」を再利用
- 少量の自前データでも高精度な分類が可能


In [ ]:
# 事前学習済み ResNet-50 を読み込む（ImageNet 重み）
backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

# 最後の全結合層（fc）を Identity に置換して特徴ベクトルを取り出す
backbone.fc = nn.Identity()

# 重みを固定（今回は特徴抽出のみ。ファインチューニングは行わない）
for param in backbone.parameters():
    param.requires_grad = False

backbone = backbone.to(device)
backbone.eval()

# 出力次元の確認
dummy = torch.zeros(1, 3, 224, 224).to(device)
with torch.no_grad():
    feat = backbone(dummy)
print(f"特徴ベクトルの次元数: {feat.shape[1]}")  # ResNet-50 では 2048
print("モデルの準備完了（重みは固定）")


## 5. img2feat ライブラリの紹介（参考）

授業では `img2feat` という便利ライブラリを使う方法も紹介されています。内部では上記と同様の特徴抽出を1行で行えます。

```python
# img2feat を使う場合（参考）
# !pip install img2feat
# from img2feat import Img2Feat
# extractor = Img2Feat(model_name='resnet50')
# features = extractor.transform(image_paths)
```

このノートブックでは PyTorch で同等の処理を実装し、内部の仕組みを理解しながら進めます。


In [ ]:
# 画像の前処理（ImageNet の標準前処理）
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],  # ImageNet の平均
        std=[0.229, 0.224, 0.225],   # ImageNet の標準偏差
    ),
])

print("前処理パイプライン:")
print("  Resize(256) → CenterCrop(224) → ToTensor → Normalize(ImageNet)")


In [ ]:
def extract_features(image_dir, model, preprocess, device, batch_size=32):
    """ディレクトリ内の画像から特徴ベクトルを抽出する。

    Parameters
    ----------
    image_dir : str
        画像が格納されたルートディレクトリ（サブフォルダ=クラス名）
    model : nn.Module
        特徴抽出モデル（最終 FC を除いたもの）
    preprocess : transforms.Compose
        前処理パイプライン
    device : torch.device
        計算デバイス

    Returns
    -------
    features : np.ndarray, shape (N, D)
    labels : np.ndarray, shape (N,) — クラス名文字列
    """
    # クラス名をサブフォルダ名から取得
    class_names = sorted([
        d for d in os.listdir(image_dir)
        if os.path.isdir(os.path.join(image_dir, d))
    ])

    all_features = []
    all_labels   = []

    for cls in class_names:
        cls_dir = os.path.join(image_dir, cls)
        img_paths = (
            glob.glob(os.path.join(cls_dir, '*.jpg')) +
            glob.glob(os.path.join(cls_dir, '*.jpeg')) +
            glob.glob(os.path.join(cls_dir, '*.png'))
        )

        cls_feats = []
        # バッチ処理で特徴抽出
        for i in range(0, len(img_paths), batch_size):
            batch_paths = img_paths[i:i+batch_size]
            batch_tensors = []
            for p in batch_paths:
                img = Image.open(p).convert('RGB')
                batch_tensors.append(preprocess(img))
            batch = torch.stack(batch_tensors).to(device)

            with torch.no_grad():
                feats = model(batch).cpu().numpy()
            cls_feats.append(feats)

        if cls_feats:
            cls_feats = np.vstack(cls_feats)
            all_features.append(cls_feats)
            all_labels.extend([cls] * len(cls_feats))

        print(f"  {cls}: {len(img_paths)} 枚 → 特徴 {cls_feats.shape if cls_feats else '(なし)'}")

    return np.vstack(all_features), np.array(all_labels)


print("特徴抽出関数の準備完了。次のセルで実行します。")


In [ ]:
# 訓練データの特徴抽出
print("訓練データの特徴抽出:")
X_train, y_train = extract_features(
    os.path.join(DATASET_ROOT, 'train'), backbone, preprocess, device
)

print()
print("テストデータの特徴抽出:")
X_test, y_test = extract_features(
    os.path.join(DATASET_ROOT, 'test'), backbone, preprocess, device
)

print()
print(f"訓練特徴量: {X_train.shape}  ラベル: {y_train.shape}")
print(f"テスト特徴量: {X_test.shape}  ラベル: {y_test.shape}")
print(f"クラス一覧: {sorted(set(y_train))}")


## 6. ロジスティック回帰による分類

抽出した特徴ベクトルを入力として、scikit-learn の LogisticRegression（ロジスティック回帰）で分類します。

なぜロジスティック回帰？
- 特徴量は CNN が抽出した高品質なベクトル（2048 次元）
- すでに線形分離可能な状態に近い
- 計算が速く、過学習しにくい（正則化 `C` パラメータで調整）


In [ ]:
# ラベルのエンコーディング（文字列 → 整数）
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

print(f"クラスエンコーディング: {dict(zip(le.classes_, le.transform(le.classes_)))}")


In [ ]:
# ロジスティック回帰の学習
# C: 正則化の逆数。小さいほど強い正則化（過学習を防ぐ）
clf = LogisticRegression(
    C=1.0,               # 正則化パラメータ
    max_iter=1000,       # 最大イテレーション数
    random_state=SEED,
    multi_class='multinomial',  # 多クラス分類
    solver='lbfgs',
    n_jobs=-1            # 並列処理
)

clf.fit(X_train, y_train_enc)
print("ロジスティック回帰の学習完了")


In [ ]:
# 評価
y_pred = clf.predict(X_test)

test_acc = accuracy_score(y_test_enc, y_pred)
print(f"テスト精度 (Test Accuracy): {test_acc * 100:.2f}%")
print()
print("クラス別レポート:")
print(classification_report(
    y_test_enc, y_pred,
    target_names=le.classes_,
    digits=3
))


## 7. 混同行列の可視化

混同行列（Confusion Matrix）は、どのクラスが何に間違えられているかを示します。対角成分が正解、それ以外が誤分類です。


In [ ]:
cm = confusion_matrix(y_test_enc, y_pred)

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=le.classes_, yticklabels=le.classes_,
    ax=ax
)
ax.set_xlabel("予測ラベル (Predicted)", fontsize=12)
ax.set_ylabel("正解ラベル (True)", fontsize=12)
ax.set_title("混同行列 (Confusion Matrix)", fontsize=13)
plt.tight_layout()
plt.show()


## 8. 特徴空間の可視化（t-SNE）

高次元の特徴ベクトル（2048 次元）を 2 次元に圧縮して可視化します。t-SNE（t-distributed Stochastic Neighbor Embedding）という手法で、同じクラスの点が近くに集まっているか確認します。


In [ ]:
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA

# まず PCA で 50 次元に圧縮してから t-SNE を適用（高速化）
n_samples = min(500, len(X_test))
idx = np.random.choice(len(X_test), n_samples, replace=False)
X_subset = X_test[idx]
y_subset = y_test[idx]

print(f"t-SNE 可視化 ({n_samples} サンプル)...")

# PCA で次元削減（前処理）
pca = PCA(n_components=50, random_state=SEED)
X_pca = pca.fit_transform(X_subset)

# t-SNE で 2 次元に圧縮
tsne = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
X_2d = tsne.fit_transform(X_pca)

# 散布図
fig, ax = plt.subplots(figsize=(10, 7))
classes = sorted(set(y_subset))
colors = plt.cm.tab10(np.linspace(0, 1, len(classes)))
for cls, color in zip(classes, colors):
    mask = y_subset == cls
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], label=cls, alpha=0.7, s=20, color=color)
ax.set_title("特徴空間の t-SNE 可視化（テストデータ）", fontsize=13)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.grid(True, alpha=0.2)
plt.tight_layout()
plt.show()
print("→ 同じクラスの点が塊（クラスタ）を形成していれば転移学習が成功しています")


## 9. 正則化パラメータ C の効果を比較する

ロジスティック回帰の `C` パラメータ（正則化の強さ）を変えてテスト精度への影響を確認します。


In [ ]:
C_values = [0.001, 0.01, 0.1, 1.0, 10.0, 100.0]
results = []

for C in C_values:
    clf_c = LogisticRegression(
        C=C, max_iter=1000, random_state=SEED,
        multi_class='multinomial', solver='lbfgs', n_jobs=-1
    )
    clf_c.fit(X_train, y_train_enc)
    train_acc = clf_c.score(X_train, y_train_enc)
    test_acc  = clf_c.score(X_test, y_test_enc)
    results.append({"C": C, "train_acc": train_acc, "test_acc": test_acc})
    print(f"  C={C:>8}: Train={train_acc*100:5.2f}%  Test={test_acc*100:5.2f}%")

# 可視化
Cs = [r["C"] for r in results]
train_accs = [r["train_acc"] * 100 for r in results]
test_accs  = [r["test_acc"]  * 100 for r in results]

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(Cs, train_accs, 'o-', label="Train Acc", color="royalblue")
ax.semilogx(Cs, test_accs,  's-', label="Test Acc",  color="tomato")
ax.set_xlabel("正則化パラメータ C (log scale)")
ax.set_ylabel("Accuracy (%)")
ax.set_title("正則化パラメータ C とテスト精度の関係")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## 10. 試してみよう（課題）

### 課題 1: 別の事前学習済みモデルを使う
- `models.resnet50` を `models.resnet18` や `models.efficientnet_b0` に変えてみましょう
- 特徴ベクトルの次元数と精度はどう変わりますか？

### 課題 2: 自前データで実験する
- `USE_DEMO_DATA = False` にして自分で用意した画像を試してみましょう
- クラス数や 1 クラスあたりの枚数を変えると精度はどう変わりますか？

### 課題 3: 最終レポートに向けて
- 授業課題3・最終レポート（締切: 2026-07-30）では、この枠組みを使って  自分が設定した問題（分類タスク）を解きます
- データセットの選定・精度報告・考察の書き方を事前に確認しておきましょう
